In [ ]:
# ============================================================
# Expert-Level Object Detection + Tracking Project for Colab
# Detector: YOLO or Faster R-CNN
# Tracker : Deep SORT
# Output  : Live annotated frames + saved MP4
# ============================================================

!pip -q install -U ultralytics deep-sort-realtime opencv-python-headless torchvision

import os
import cv2
import time
import warnings
import numpy as np
import torch
import torchvision

from collections import defaultdict, deque
from IPython.display import display, clear_output, Video
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

warnings.filterwarnings("ignore")

# -------------------------------
# Colab Helpers
# -------------------------------
IN_COLAB = False
try:
    from google.colab import files
    from google.colab.patches import cv2_imshow
    IN_COLAB = True
except:
    pass

# -------------------------------
# Configuration
# -------------------------------
CONFIG = {
    # Detector options: "yolo" or "fasterrcnn"
    "DETECTOR_BACKEND": "yolo",

    # YOLO weights: try yolo11n.pt / yolo11s.pt / yolo11m.pt
    "YOLO_WEIGHTS": "yolo11n.pt",

    "CONFIDENCE": 0.35,
    "IOU": 0.45,
    "IMG_SIZE": 960,

    # None = track all classes
    # Example: ["person", "car", "bus", "truck"]
    "TRACK_CLASSES": None,

    # In Colab, use "upload"
    # For local Jupyter, you can use 0 for webcam or a file path
    "SOURCE": "upload",

    "OUTPUT_PATH": "tracked_output.mp4",
    "SHOW_LIVE": True,
    "DISPLAY_EVERY_N_FRAMES": 1,
    "MAX_TRAIL": 30,

    # Deep SORT params
    "MAX_AGE": 30,
    "N_INIT": 3,
    "NN_BUDGET": 100,
    "MAX_IOU_DISTANCE": 0.7,
    "EMBEDDER": "mobilenet",

    # Drawing
    "ANNOTATION_SCALE": 0.7,
    "BOX_THICKNESS": 2,
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# -------------------------------
# COCO Class Names for Faster R-CNN
# -------------------------------
COCO_INSTANCE_CATEGORY_NAMES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A', 'N/A',
    'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
    'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table',
    'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A',
    'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

# -------------------------------
# Utilities
# -------------------------------
def color_from_id(track_id: int):
    rng = np.random.default_rng(seed=int(track_id) + 12345)
    return tuple(int(x) for x in rng.integers(40, 255, size=3))

def class_allowed(class_name):
    if CONFIG["TRACK_CLASSES"] is None:
        return True
    return class_name in set(CONFIG["TRACK_CLASSES"])

def resolve_source():
    src = CONFIG["SOURCE"]
    if src == "upload":
        if not IN_COLAB:
            raise ValueError("SOURCE='upload' is intended for Google Colab.")
        print("Upload a video file for detection + tracking...")
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("No file uploaded.")
        return next(iter(uploaded.keys()))
    return src

# -------------------------------
# Detector Backends
# -------------------------------
class YOLODetector:
    def __init__(self, weights, conf, iou, imgsz):
        self.model = YOLO(weights)
        self.conf = conf
        self.iou = iou
        self.imgsz = imgsz
        self.names = self.model.names

    def detect(self, frame):
        results = self.model.predict(
            source=frame,
            conf=self.conf,
            iou=self.iou,
            imgsz=self.imgsz,
            device=0 if DEVICE == "cuda" else "cpu",
            half=(DEVICE == "cuda"),
            verbose=False
        )

        result = results[0]
        detections = []

        if result.boxes is None or len(result.boxes) == 0:
            return detections

        boxes = result.boxes.xyxy.detach().cpu().numpy()
        scores = result.boxes.conf.detach().cpu().numpy()
        labels = result.boxes.cls.detach().cpu().numpy().astype(int)

        for box, score, label in zip(boxes, scores, labels):
            class_name = self.names[int(label)]
            if not class_allowed(class_name):
                continue

            x1, y1, x2, y2 = box.astype(int)
            detections.append({
                "bbox_xyxy": [x1, y1, x2, y2],
                "confidence": float(score),
                "class_id": int(label),
                "class_name": str(class_name)
            })
        return detections


class FasterRCNNDetector:
    def __init__(self, conf):
        weights = torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT
        self.model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=weights)
        self.model.to(DEVICE).eval()
        self.conf = conf
        self.names = COCO_INSTANCE_CATEGORY_NAMES

    def detect(self, frame):
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        tensor = torch.from_numpy(rgb).permute(2, 0, 1).float().div(255.0).to(DEVICE)

        with torch.no_grad():
            pred = self.model([tensor])[0]

        boxes = pred["boxes"].detach().cpu().numpy()
        scores = pred["scores"].detach().cpu().numpy()
        labels = pred["labels"].detach().cpu().numpy().astype(int)

        detections = []
        for box, score, label in zip(boxes, scores, labels):
            if float(score) < self.conf:
                continue

            class_name = self.names[int(label)] if int(label) < len(self.names) else str(label)
            if class_name == "N/A" or not class_allowed(class_name):
                continue

            x1, y1, x2, y2 = box.astype(int)
            detections.append({
                "bbox_xyxy": [x1, y1, x2, y2],
                "confidence": float(score),
                "class_id": int(label),
                "class_name": str(class_name)
            })
        return detections

# -------------------------------
# Deep SORT Tracker
# -------------------------------
class DeepSortTracker:
    def __init__(self):
        self.tracker = DeepSort(
            max_age=CONFIG["MAX_AGE"],
            n_init=CONFIG["N_INIT"],
            nn_budget=CONFIG["NN_BUDGET"],
            max_iou_distance=CONFIG["MAX_IOU_DISTANCE"],
            embedder=CONFIG["EMBEDDER"]
        )

    def update(self, frame, detections):
        ds_detections = []
        for det in detections:
            x1, y1, x2, y2 = det["bbox_xyxy"]
            w = max(0, x2 - x1)
            h = max(0, y2 - y1)
            ds_detections.append((
                [x1, y1, w, h],
                det["confidence"],
                det["class_name"]
            ))

        tracks = self.tracker.update_tracks(ds_detections, frame=frame)
        tracked = []

        for track in tracks:
            if not track.is_confirmed():
                continue

            l, t, r, b = track.to_ltrb()

            cls_name = None
            if hasattr(track, "get_det_class"):
                cls_name = track.get_det_class()
            if cls_name is None and hasattr(track, "det_class"):
                cls_name = track.det_class
            if cls_name is None:
                cls_name = "object"

            tracked.append({
                "track_id": int(track.track_id),
                "bbox_xyxy": [int(l), int(t), int(r), int(b)],
                "class_name": str(cls_name)
            })

        return tracked

# -------------------------------
# Visualizer
# -------------------------------
class Visualizer:
    def __init__(self, max_trail=30):
        self.trails = defaultdict(lambda: deque(maxlen=max_trail))
        self.unique_ids = set()

    def draw(self, frame, tracks, fps, frame_idx):
        canvas = frame.copy()
        per_class_counts = defaultdict(int)

        for tr in tracks:
            x1, y1, x2, y2 = tr["bbox_xyxy"]
            track_id = tr["track_id"]
            class_name = tr["class_name"]

            self.unique_ids.add(track_id)
            per_class_counts[class_name] += 1

            color = color_from_id(track_id)
            cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)
            self.trails[track_id].append((cx, cy))

            # Bounding box
            cv2.rectangle(canvas, (x1, y1), (x2, y2), color, CONFIG["BOX_THICKNESS"])

            # Label background
            label = f"{class_name} | ID {track_id}"
            (tw, th), _ = cv2.getTextSize(
                label, cv2.FONT_HERSHEY_SIMPLEX, CONFIG["ANNOTATION_SCALE"], 2
            )
            cv2.rectangle(canvas, (x1, max(0, y1 - th - 10)), (x1 + tw + 10, y1), color, -1)
            cv2.putText(
                canvas, label, (x1 + 5, y1 - 6),
                cv2.FONT_HERSHEY_SIMPLEX, CONFIG["ANNOTATION_SCALE"],
                (0, 0, 0), 2, cv2.LINE_AA
            )

            # Trajectory trail
            pts = list(self.trails[track_id])
            for i in range(1, len(pts)):
                thickness = max(1, int(4 * (i / len(pts))))
                cv2.line(canvas, pts[i - 1], pts[i], color, thickness)

        # Info panel
        lines = [
            f"Detector: {CONFIG['DETECTOR_BACKEND']}",
            f"Tracker: Deep SORT",
            f"FPS: {fps:.2f}",
            f"Frame: {frame_idx}",
            f"Active Tracks: {len(tracks)}",
            f"Unique IDs Seen: {len(self.unique_ids)}",
        ]
        for k, v in sorted(per_class_counts.items()):
            lines.append(f"{k}: {v}")

        y = 28
        for line in lines:
            cv2.putText(
                canvas, line, (12, y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.72,
                (20, 255, 20), 2, cv2.LINE_AA
            )
            y += 28

        return canvas

# -------------------------------
# Build Pipeline
# -------------------------------
if CONFIG["DETECTOR_BACKEND"].lower() == "yolo":
    detector = YOLODetector(
        weights=CONFIG["YOLO_WEIGHTS"],
        conf=CONFIG["CONFIDENCE"],
        iou=CONFIG["IOU"],
        imgsz=CONFIG["IMG_SIZE"]
    )
elif CONFIG["DETECTOR_BACKEND"].lower() == "fasterrcnn":
    detector = FasterRCNNDetector(conf=CONFIG["CONFIDENCE"])
else:
    raise ValueError("DETECTOR_BACKEND must be 'yolo' or 'fasterrcnn'.")

tracker = DeepSortTracker()
visualizer = Visualizer(max_trail=CONFIG["MAX_TRAIL"])

# -------------------------------
# Open Source
# -------------------------------
source = resolve_source()
cap = cv2.VideoCapture(source)
if not cap.isOpened():
    raise RuntimeError(f"Could not open source: {source}")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 1280)
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 720)
fps_in = float(cap.get(cv2.CAP_PROP_FPS) or 25.0)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(CONFIG["OUTPUT_PATH"], fourcc, fps_in, (width, height))

print(f"Opened source: {source}")
print(f"Resolution: {width}x{height} | Input FPS: {fps_in:.2f}")

# -------------------------------
# Main Loop
# -------------------------------
frame_idx = 0
fps_smooth = 0.0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    start = time.time()

    detections = detector.detect(frame)
    tracks = tracker.update(frame, detections)

    elapsed = time.time() - start
    inst_fps = 1.0 / max(elapsed, 1e-6)
    fps_smooth = inst_fps if fps_smooth == 0 else (0.9 * fps_smooth + 0.1 * inst_fps)

    annotated = visualizer.draw(frame, tracks, fps_smooth, frame_idx)
    writer.write(annotated)

    if CONFIG["SHOW_LIVE"] and IN_COLAB and frame_idx % CONFIG["DISPLAY_EVERY_N_FRAMES"] == 0:
        clear_output(wait=True)
        cv2_imshow(annotated)
        print(f"Processing frame {frame_idx} | Smoothed FPS: {fps_smooth:.2f}")

    frame_idx += 1

cap.release()
writer.release()

print(f"\nDone. Output saved to: {CONFIG['OUTPUT_PATH']}")

if IN_COLAB:
    display(Video(CONFIG["OUTPUT_PATH"], embed=True))
